In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

print("Libraries imported")
print("NumPy version:", np.__version__)


In [ ]:
# Quantization configuration
SCALE = 512
SHIFT_BITS = 9
DUR_SHIFT = 20
RATE_SHIFT = 10
DUR_SCALE = 0.001
MAX_DEN = 512
RECIP_SHIFT = 16

def quantize_weight_int16(w):
    w_scaled = w * SCALE
    w_clipped = np.clip(w_scaled, -32768, 32767)
    return np.round(w_clipped).astype(np.int16)

def quantize_bias_int32(b):
    b_scaled = b * SCALE
    b_clipped = np.clip(b_scaled, -2**31, 2**31 - 1)
    return np.round(b_clipped).astype(np.int32)

def quantize_output_bias_int32(b):
    b_scaled = b * (SCALE * SCALE)
    b_clipped = np.clip(b_scaled, -2**31, 2**31 - 1)
    return np.round(b_clipped).astype(np.int32)

def quantize_features(X):
    return np.clip(X, 0, SCALE).astype(np.int16)

print(f"Quantization scale: {SCALE} (shift={SHIFT_BITS})")


In [ ]:
# Load dataset and build engineered P4 fields
train_df = pd.read_csv('../../../../data/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('../../../../data/UNSW_NB15_testing-set.csv')

proto_map = {
    'tcp': 6,
    'udp': 17,
    'icmp': 1,
    'arp': 255,
}

train_df['proto_encoded'] = train_df['proto'].map(lambda x: proto_map.get(str(x).lower(), 255))
test_df['proto_encoded'] = test_df['proto'].map(lambda x: proto_map.get(str(x).lower(), 255))

for df in (train_df, test_df):
    df['totpkts'] = df['spkts'] + df['dpkts']
    df['totbytes'] = df['sbytes'] + df['dbytes']

recip_pkt = np.zeros(MAX_DEN + 1, dtype=np.int64)
recip_dur = np.zeros(MAX_DEN + 1, dtype=np.int64)
for d in range(1, MAX_DEN + 1):
    recip_pkt[d] = (1 << RECIP_SHIFT) // d
    recip_dur[d] = (1 << (RECIP_SHIFT + RATE_SHIFT)) // d

def add_mean_and_rate_features(df):
    spkts_idx = np.clip(df['spkts'].astype('int64').to_numpy(), 1, MAX_DEN)
    dpkts_idx = np.clip(df['dpkts'].astype('int64').to_numpy(), 1, MAX_DEN)
    sbytes = df['sbytes'].astype('int64').to_numpy()
    dbytes = df['dbytes'].astype('int64').to_numpy()

    df['smean'] = (sbytes * recip_pkt[spkts_idx]) >> RECIP_SHIFT
    df['dmean'] = (dbytes * recip_pkt[dpkts_idx]) >> RECIP_SHIFT

    dur_ns = df['dur'].to_numpy() * 1e9 * DUR_SCALE
    dur_scaled = (dur_ns // (1 << DUR_SHIFT)).astype('int64')
    dur_idx = np.clip(dur_scaled, 1, MAX_DEN)
    totpkts = df['totpkts'].astype('int64').to_numpy()
    df['rate'] = (totpkts * recip_dur[dur_idx]) >> RECIP_SHIFT

add_mean_and_rate_features(train_df)
add_mean_and_rate_features(test_df)

print(f"Training samples: {len(train_df):,}")
print(f"Test samples: {len(test_df):,}")


In [ ]:
# Prepare labels, augmentation, and P4 scaling
AUGMENT_PARTIAL = True
PARTIAL_SAMPLES_PER_FLOW = 1
PARTIAL_FRAC_MIN = 0.2
PARTIAL_FRAC_MAX = 0.9

def make_partial_snapshots(df, per_flow, frac_min, frac_max, seed=42):
    rng = np.random.default_rng(seed)
    base = df.loc[df.index.repeat(per_flow)].copy()
    fracs = rng.uniform(frac_min, frac_max, size=len(base))

    base['dur'] = base['dur'] * fracs
    base['spkts'] = np.maximum(1, (base['spkts'] * fracs).astype(int))
    base['dpkts'] = (base['dpkts'] * fracs).astype(int)
    base.loc[base['dpkts'] < 1, 'dpkts'] = 0

    base['sbytes'] = (base['sbytes'] * fracs).astype(int)
    base['dbytes'] = (base['dbytes'] * fracs).astype(int)
    base.loc[base['dpkts'] == 0, 'dbytes'] = 0

    base['totpkts'] = base['spkts'] + base['dpkts']
    base['totbytes'] = base['sbytes'] + base['dbytes']

    spkts_idx = np.clip(base['spkts'].astype('int64').to_numpy(), 1, MAX_DEN)
    dpkts_idx = np.clip(base['dpkts'].astype('int64').to_numpy(), 1, MAX_DEN)
    sbytes = base['sbytes'].astype('int64').to_numpy()
    dbytes = base['dbytes'].astype('int64').to_numpy()
    base['smean'] = (sbytes * recip_pkt[spkts_idx]) >> RECIP_SHIFT
    base['dmean'] = (dbytes * recip_pkt[dpkts_idx]) >> RECIP_SHIFT

    dur_ns = base['dur'].to_numpy() * 1e9 * DUR_SCALE
    dur_scaled = (dur_ns // (1 << DUR_SHIFT)).astype('int64')
    dur_idx = np.clip(dur_scaled, 1, MAX_DEN)
    totpkts = base['totpkts'].astype('int64').to_numpy()
    base['rate'] = (totpkts * recip_dur[dur_idx]) >> RECIP_SHIFT
    return base

if AUGMENT_PARTIAL:
    partial_df = make_partial_snapshots(
        train_df,
        per_flow=PARTIAL_SAMPLES_PER_FLOW,
        frac_min=PARTIAL_FRAC_MIN,
        frac_max=PARTIAL_FRAC_MAX,
    )
    train_df = pd.concat([train_df, partial_df], ignore_index=True)
    print(f"Partial-flow augmentation: +{len(partial_df):,} rows")

y_train = train_df['label'].to_numpy()
y_test = test_df['label'].to_numpy()

def p4_scale_features(X, feature_names):
    X_scaled = X.astype(np.int64).copy()

    for i, feat in enumerate(feature_names):
        col = X_scaled[:, i]
        if feat == 'proto_encoded':
            X_scaled[:, i] = col << 4
        elif feat in ['sttl', 'dttl']:
            X_scaled[:, i] = col << 1
        elif feat in ['swin', 'dwin']:
            X_scaled[:, i] = col >> 7
        elif feat in ['sbytes', 'dbytes', 'totbytes']:
            X_scaled[:, i] = np.where(col > 24, (col - 24) >> 6, 0)
        elif feat in ['spkts', 'dpkts', 'totpkts']:
            X_scaled[:, i] = np.where(col > 0, (col - 1) << 2, 0)
        elif feat == 'dur':
            dur_ns = (col.astype(np.float64) * 1e9 * DUR_SCALE).astype(np.int64)
            X_scaled[:, i] = dur_ns >> DUR_SHIFT
        elif feat == 'rate':
            X_scaled[:, i] = col
        elif feat in ['smean', 'dmean']:
            X_scaled[:, i] = np.where(col > 24, (col - 24) >> 1, 0)

    return np.clip(X_scaled, 0, SCALE)

normal_count = np.sum(y_train == 0)
attack_count = np.sum(y_train == 1)
total_count = len(y_train)

print(f"Training labels: {total_count:,}")
print(f"Normal: {normal_count:,}")
print(f"Attack: {attack_count:,}")


In [ ]:
class QuantizedNeuralNetwork:
    """Quantized 9-32-1 neural network aligned with the P4 dataplane."""

    def __init__(self, verbose=True):
        self.input_size = 9   # dur, rate, proto, dpkts, sttl, dwin, dttl, smean, sbytes
        self.hidden_size = 32
        self.output_size = 1

        self.W1 = np.random.randn(self.input_size, self.hidden_size) * 1.0
        self.b1 = np.zeros(self.hidden_size)
        self.W2 = np.random.randn(self.hidden_size, self.output_size) * 1.0
        self.b2 = np.zeros(self.output_size)

        total_params = (
            self.input_size * self.hidden_size
            + self.hidden_size
            + self.hidden_size * self.output_size
            + self.output_size
        )

        if verbose:
            print("Neural Network Architecture")
            print(f"  Input layer:  {self.input_size} features")
            print(f"  Hidden layer: {self.hidden_size} neurons")
            print(f"  Output layer: {self.output_size} neuron")
            print(f"  Total parameters: {total_params}")

    def forward_training(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(self.z1, 0)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = 1 / (1 + np.exp(-self.z2))
        return self.a2

    def forward_quantized(self, X_quantized):
        W1_int = quantize_weight_int16(self.W1)
        b1_int = quantize_bias_int32(self.b1)
        W2_int = quantize_weight_int16(self.W2)
        b2_int = quantize_output_bias_int32(self.b2)

        z1_int = (X_quantized.astype(np.int32) @ W1_int.astype(np.int32)) + (b1_int * SCALE)
        a1_int = np.maximum(z1_int, 0)
        a1_scaled = (a1_int >> SHIFT_BITS).astype(np.int16)
        z2_int = (a1_scaled.astype(np.int32) @ W2_int.astype(np.int32)) + b2_int

        self.z1_int = z1_int
        self.a1_int = a1_int
        self.a1_scaled = a1_scaled
        self.z2_int = z2_int
        return z2_int

    def predict_quantized(self, X_quantized, threshold):
        z2_int = self.forward_quantized(X_quantized)
        return (z2_int >= threshold).astype(int).reshape(-1)

model = QuantizedNeuralNetwork()


In [ ]:
# Training helpers
def binary_cross_entropy(y_true, y_pred, class_weight_0=1.0, class_weight_1=1.0):
    eps = 1e-7
    y_true = y_true.reshape(-1, 1)
    y_pred = np.clip(y_pred, eps, 1 - eps)
    weights = np.where(y_true == 1, class_weight_1, class_weight_0)
    loss = -(weights * (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)))
    return loss.mean()

def train_epoch(model, X, y, batch_size, lr, class_weight_0, class_weight_1):
    n = X.shape[0]
    idx = np.random.permutation(n)
    X = X[idx]
    y = y[idx]
    total_loss = 0.0

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        xb = X[start:end]
        yb = y[start:end].reshape(-1, 1)

        z1 = xb @ model.W1 + model.b1
        a1 = np.maximum(z1, 0)
        z2 = a1 @ model.W2 + model.b2
        a2 = 1 / (1 + np.exp(-z2))

        loss = binary_cross_entropy(yb, a2, class_weight_0, class_weight_1)
        total_loss += loss * (end - start)

        weights = np.where(yb == 1, class_weight_1, class_weight_0)
        dz2 = (a2 - yb) * weights / (end - start)
        dW2 = a1.T @ dz2
        db2 = dz2.sum(axis=0)

        da1 = dz2 @ model.W2.T
        dz1 = da1 * (z1 > 0)
        dW1 = xb.T @ dz1
        db1 = dz1.sum(axis=0)

        model.W2 -= lr * dW2
        model.b2 -= lr * db2
        model.W1 -= lr * dW1
        model.b1 -= lr * db1

    return total_loss / n

def evaluate_model(model, X, y):
    y = y.reshape(-1, 1)
    z2 = model.forward_training(X)
    y_pred = (z2 >= 0.5).astype(int).reshape(-1)
    y_true = y.reshape(-1)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return acc, prec, rec, f1


In [ ]:
# Build matrices for the final selected feature set
features = ['dur', 'rate', 'proto', 'dpkts', 'sttl', 'dwin', 'dttl', 'smean', 'sbytes']
feature_cols = [('proto_encoded' if f == 'proto' else f) for f in features]

X_train_raw = train_df[feature_cols].to_numpy()
X_test_raw = test_df[feature_cols].to_numpy()

X_train_p4 = p4_scale_features(X_train_raw, feature_cols)
X_test_p4 = p4_scale_features(X_test_raw, feature_cols)

X_train_scaled = X_train_p4 / SCALE
X_test_scaled = X_test_p4 / SCALE

X_train_quantized = X_train_p4.astype(np.int16)
X_test_quantized = X_test_p4.astype(np.int16)

print("Using final selected features:", features)
print("Training matrix:", X_train_scaled.shape)
print("Test matrix:", X_test_scaled.shape)


In [ ]:
# Training configuration
batch_size = 64
learning_rate = 0.001
epochs = 100
early_stopping_patience = 10

NORMAL_WEIGHT_MULTIPLIER = 4.0
ATTACK_WEIGHT_MULTIPLIER = 1.0

class_counts = np.bincount(y_train)
total_samples = len(y_train)
class_weight_0 = (total_samples / (2 * class_counts[0])) * NORMAL_WEIGHT_MULTIPLIER
class_weight_1 = (total_samples / (2 * class_counts[1])) * ATTACK_WEIGHT_MULTIPLIER

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print(f"Class weight normal: {class_weight_0:.3f}")
print(f"Class weight attack: {class_weight_1:.3f}")
print(f"Training split: {len(X_train_split):,}")
print(f"Validation split: {len(X_val_split):,}")
print(f"Test split: {len(X_test_scaled):,}")


In [ ]:
# Train with early stopping
print("Starting quantized training")

best_val_f1 = -np.inf
patience_counter = 0
best_W1 = model.W1.copy()
best_b1 = model.b1.copy()
best_W2 = model.W2.copy()
best_b2 = model.b2.copy()

for epoch in range(epochs):
    train_loss = train_epoch(
        model,
        X_train_split,
        y_train_split,
        batch_size,
        learning_rate,
        class_weight_0,
        class_weight_1,
    )

    train_acc, train_prec, train_rec, train_f1 = evaluate_model(model, X_train_split, y_train_split)
    val_acc, val_prec, val_rec, val_f1 = evaluate_model(model, X_val_split, y_val_split)

    if (epoch + 1) % 10 == 0 or epoch < 5:
        print(
            f"Epoch {epoch + 1:3d}/{epochs}: "
            f"loss={train_loss:.4f} | "
            f"train_f1={train_f1:.4f} | "
            f"val_f1={val_f1:.4f}"
        )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        best_W1 = model.W1.copy()
        best_b1 = model.b1.copy()
        best_W2 = model.W2.copy()
        best_b2 = model.b2.copy()
    else:
        patience_counter += 1

    if patience_counter >= early_stopping_patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.W1 = best_W1
model.b1 = best_b1
model.W2 = best_W2
model.b2 = best_b2

val_acc, val_prec, val_rec, val_f1 = evaluate_model(model, X_val_split, y_val_split)
print("Final validation metrics")
print(f"  Accuracy: {val_acc:.4f}")
print(f"  Precision: {val_prec:.4f}")
print(f"  Recall: {val_rec:.4f}")
print(f"  F1: {val_f1:.4f}")


In [ ]:
# Calibrate decision threshold and evaluate on the test set
z2_test_int = model.forward_quantized(X_test_quantized)
X_val_quantized = quantize_features(X_val_split * SCALE)
z2_val = model.forward_quantized(X_val_quantized)

TARGET_FPR = 0.20
val_threshold_range = np.linspace(z2_val.min(), z2_val.max(), 400)

best_candidate = None
fallback_candidate = None

for thresh in val_threshold_range:
    y_pred_val = (z2_val >= thresh).astype(int).flatten()
    tn, fp, fn, tp = confusion_matrix(y_val_split, y_pred_val).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    candidate_score = (recall, precision, accuracy, -fpr)
    fallback_score = (-fpr, recall, precision, accuracy)

    if fpr <= TARGET_FPR and (best_candidate is None or candidate_score > best_candidate[0]):
        best_candidate = (candidate_score, thresh, accuracy, precision, recall, fpr)

    if fallback_candidate is None or fallback_score > fallback_candidate[0]:
        fallback_candidate = (fallback_score, thresh, accuracy, precision, recall, fpr)

if best_candidate is None:
    print("No threshold met the target FPR; using the minimum-FPR fallback.")
    _, best_threshold, val_accuracy, val_precision, val_recall, val_fpr = fallback_candidate
else:
    _, best_threshold, val_accuracy, val_precision, val_recall, val_fpr = best_candidate

best_threshold = int(best_threshold)
print(f"Selected threshold: {best_threshold}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation precision: {val_precision:.4f}")
print(f"Validation recall: {val_recall:.4f}")
print(f"Validation FPR: {val_fpr:.4f}")

y_pred_test = (z2_test_int >= best_threshold).astype(int).flatten()
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

final_accuracy = (tp + tn) / (tp + tn + fp + fn)
final_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
final_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
final_f1 = (
    2 * final_precision * final_recall / (final_precision + final_recall)
    if (final_precision + final_recall) > 0
    else 0.0
)

print("Test metrics")
print(f"  Accuracy: {final_accuracy:.4f}")
print(f"  Precision: {final_precision:.4f}")
print(f"  Recall: {final_recall:.4f}")
print(f"  F1: {final_f1:.4f}")
print("Confusion matrix")
print(f"  TN={tn} FP={fp}")
print(f"  FN={fn} TP={tp}")


In [ ]:
# Export quantized model for P4 deployment
import json
import os
from datetime import datetime

W1_deploy = quantize_weight_int16(model.W1)
b1_deploy = quantize_bias_int32(model.b1)
W2_deploy = quantize_weight_int16(model.W2)
b2_deploy = quantize_output_bias_int32(model.b2)
threshold_deploy = int(best_threshold)

candidate_features = [
    'proto', 'sttl', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'totpkts',
    'totbytes', 'dur', 'rate', 'smean', 'dmean', 'dttl', 'swin', 'dwin'
]
feature_map_list = [candidate_features.index(f) for f in features]
feature_mask_bits = (1 << len(features)) - 1

total_params = (
    model.input_size * model.hidden_size
    + model.hidden_size
    + model.hidden_size * model.output_size
    + model.output_size
)

model_export = {
    "model_metadata": {
        "name": "p4_neural_network_ids",
        "date": datetime.now().strftime("%Y-%m-%d"),
        "architecture": "9-32-1",
        "quantization_scale": int(SCALE),
        "total_parameters": int(total_params),
        "bias_scale_hidden": int(SCALE),
        "bias_scale_output": int(SCALE * SCALE),
        "partial_flow_augmentation": {
            "enabled": bool(AUGMENT_PARTIAL),
            "samples_per_flow": int(PARTIAL_SAMPLES_PER_FLOW),
            "frac_min": float(PARTIAL_FRAC_MIN),
            "frac_max": float(PARTIAL_FRAC_MAX),
        },
    },
    "layer_0": {
        "type": "dense",
        "input_size": int(model.input_size),
        "output_size": int(model.hidden_size),
        "activation": "relu",
        "weights": W1_deploy.tolist(),
        "biases": b1_deploy.tolist(),
    },
    "layer_1": {
        "type": "dense",
        "input_size": int(model.hidden_size),
        "output_size": int(model.output_size),
        "activation": "linear",
        "weights": W2_deploy.tolist(),
        "biases": b2_deploy.tolist(),
    },
    "decision": {
        "type": "threshold",
        "threshold": int(threshold_deploy),
        "rule": "z2 >= threshold ? ATTACK : NORMAL",
    },
    "feature_mask": int(feature_mask_bits),
    "feature_map": list(feature_map_list),
    "selected_features": list(features),
    "performance": {
        "test_accuracy": float(final_accuracy),
        "test_precision": float(final_precision),
        "test_recall": float(final_recall),
        "test_f1": float(final_f1),
    },
    "scaling_parameters": {
        "type": "p4_custom_no_minmax",
        "step1_p4_transforms": {
            "proto_encoded": "x << 4",
            "sttl_dttl": "x << 1",
            "sbytes_dbytes_totbytes": "(x - 24) >> 6",
            "spkts_dpkts_totpkts": "(x - 1) << 2",
            "dur": "(dur * 1e9 * DUR_SCALE) >> DUR_SHIFT",
            "rate": "(totpkts * recip_dur) >> 16",
            "smean_dmean": "(x - 24) >> 1",
            "dwin_swin": "x >> 7",
            "clip": "[0, 512]",
        },
    },
}

os.makedirs('../output', exist_ok=True)
export_filename = '../output/qat_model.json'
with open(export_filename, 'w') as f:
    json.dump(model_export, f, indent=2)

with open(export_filename, 'r') as f:
    loaded_model = json.load(f)

export_match = (
    loaded_model['layer_0']['weights'] == W1_deploy.tolist()
    and loaded_model['layer_0']['biases'] == b1_deploy.tolist()
    and loaded_model['layer_1']['weights'] == W2_deploy.tolist()
    and loaded_model['layer_1']['biases'] == b2_deploy.tolist()
    and loaded_model['decision']['threshold'] == threshold_deploy
    and loaded_model['selected_features'] == features
    and loaded_model['feature_map'] == feature_map_list
)

print(f"Model exported to {export_filename}")
print(f"Export verification: {'PASSED' if export_match else 'FAILED'}")
